In [1]:
# NOTEBOOK: 04_model.ipynb  -- cell 0 (run FIRST after restart, before any GPU work)
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
print("debug mode on")

debug mode on


In [2]:
# NOTEBOOK: 04_model.ipynb  -- cell 1 (does the model load on GPU?)
import torch
from transformers import AutoProcessor, AutoModelForTokenClassification

print("GPU available:", torch.cuda.is_available())

model_name = "microsoft/layoutlmv3-base"
processor = AutoProcessor.from_pretrained(model_name, apply_ocr=False)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=3)
model = model.to("cuda")

print("model loaded on:", next(model.parameters()).device)

/home/jovyan/receipt-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 27821.17it/s]
[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model loaded on: cuda:0


In [3]:
# NOTEBOOK: 04_model.ipynb  -- cell 2 (load labeled data, regroup into receipts)
import pandas as pd
import glob, os

# parquet was written as a folder of part files; read them all
parts = glob.glob("data/processed/labeled_words.parquet/*.parquet")
df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)

print("total word rows:", len(df))
print("receipts:", df["idx"].nunique())
print(df["label"].value_counts())
df.head()

total word rows: 14687
receipts: 800
label
OTHER    13775
TOTAL      912
Name: count, dtype: int64


,idx,text,conf,box,label
0,138,5,0.273074,"[[1543.0, 1125.0], [1580.0, 1125.0], [1580.0, ...",OTHER
1,138,5337,0.034455,"[[229.0, 1177.0], [367.0, 1177.0], [367.0, 121...",OTHER
2,138,Bacon ShIMEJI Spaghe 1.00,0.068014,"[[156.0, 1219.0], [1055.0, 1219.0], [1055.0, 1...",OTHER
3,138,ABODD,0.291582,"[[1102.0, 1212.0], [1299.0, 1212.0], [1299.0, ...",OTHER
4,138,4o008,0.227278,"[[1384.0, 1194.0], [1591.0, 1194.0], [1591.0, ...",OTHER


In [4]:
# NOTEBOOK: 04_model.ipynb  -- cell 3b (inspect one box's actual type and shape)
one = df["box"].iloc[0]
print("type:", type(one))
print("value:", one)

type: <class 'numpy.ndarray'>
value: [array([1543., 1125.]) array([1580., 1125.]) array([1580., 1157.])
 array([1543., 1157.])]


In [4]:
# NOTEBOOK: 04_model.ipynb  -- cell 3 (fixed: handle nested numpy box)
import numpy as np

def box_to_xyxy(box):
    # box is an array of 4 corner arrays: [array([x,y]), array([x,y]), ...]
    pts = np.vstack([np.asarray(p, dtype=float) for p in box])  # -> shape (4, 2)
    xmin, ymin = pts[:, 0].min(), pts[:, 1].min()
    xmax, ymax = pts[:, 0].max(), pts[:, 1].max()
    return [float(xmin), float(ymin), float(xmax), float(ymax)]

examples = []
for idx, g in df.groupby("idx"):
    words  = g["text"].astype(str).tolist()
    boxes  = [box_to_xyxy(b) for b in g["box"]]
    labels = g["label"].tolist()
    examples.append({"idx": int(idx), "words": words, "boxes": boxes, "labels": labels})

print("receipts grouped:", len(examples))
ex = examples[0]
print("receipt idx:", ex["idx"])
print("num words:", len(ex["words"]))
print("first 3 words:", ex["words"][:3])
print("first 3 boxes:", ex["boxes"][:3])
print("labels present:", set(ex["labels"]))

receipts grouped: 800
receipt idx: 0
num words: 86
first 3 words: ['Nasi', 'Campur', 'Bali']
first 3 boxes: [[300.0, 366.0, 354.0, 392.0], [362.0, 366.0, 440.0, 390.0], [446.0, 364.0, 498.0, 388.0]]
labels present: {'OTHER'}


In [6]:
# NOTEBOOK: 04_model.ipynb  -- cell 4 (feed ONE receipt through the processor)
from datasets import load_dataset
from PIL import Image
import numpy as np

# we need the receipt IMAGE too (LayoutLMv3 uses image + words + boxes)
ds = load_dataset("naver-clova-ix/cord-v2")["train"]

# pick a receipt that HAS a TOTAL label, so the example is meaningful
ex = next(e for e in examples if "TOTAL" in e["labels"])
print("using receipt idx:", ex["idx"], "| words:", len(ex["words"]))

image = ds[ex["idx"]]["image"].convert("RGB")
W, H = image.size

# LayoutLMv3 needs boxes scaled to 0..1000 relative to image size
def scale_box(b, W, H):
    x0, y0, x1, y1 = b
    return [int(1000*x0/W), int(1000*y0/H), int(1000*x1/W), int(1000*y1/H)]

boxes_scaled = [scale_box(b, W, H) for b in ex["boxes"]]

# map string labels to ids
label2id = {"OTHER": 0, "TOTAL": 1}
word_labels = [label2id[l] for l in ex["labels"]]

# run the processor (turns words+boxes+image into model inputs)
enc = processor(
    image,
    ex["words"],
    boxes=boxes_scaled,
    word_labels=word_labels,
    return_tensors="pt",
    truncation=True,
    padding="max_length",
)

print("input keys:", list(enc.keys()))
print("input_ids shape:", enc["input_ids"].shape)
print("labels shape:", enc["labels"].shape)
print("num TOTAL tokens in this receipt:", (enc["labels"] == 1).sum().item())

using receipt idx: 1 | words: 30
input keys: ['input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values']
input_ids shape: torch.Size([1, 512])
labels shape: torch.Size([1, 512])
num TOTAL tokens in this receipt: 1


In [5]:
# NOTEBOOK: 04_model.ipynb  -- cell 5 (split into train and test)
import random
random.seed(42)   # reproducible: same split every run

# only keep receipts that actually have a TOTAL to learn from
labeled = [e for e in examples if "TOTAL" in e["labels"]]
print("receipts with a TOTAL label:", len(labeled))

random.shuffle(labeled)
n_test = int(0.2 * len(labeled))     # 20% held out for testing
test_examples  = labeled[:n_test]
train_examples = labeled[n_test:]

print("train receipts:", len(train_examples))
print("test receipts:", len(test_examples))

receipts with a TOTAL label: 498
train receipts: 399
test receipts: 99


In [6]:
# NOTEBOOK: 04_model.ipynb  -- cell 6 (PyTorch dataset wrapper)
import torch
from torch.utils.data import Dataset
from datasets import load_dataset

ds_img = load_dataset("naver-clova-ix/cord-v2")["train"]
label2id = {"OTHER": 0, "TOTAL": 1}

def scale_box(b, W, H):
    x0, y0, x1, y1 = b
    return [int(1000*x0/W), int(1000*y0/H), int(1000*x1/W), int(1000*y1/H)]

class ReceiptDataset(Dataset):
    def __init__(self, examples, processor):
        self.examples = examples
        self.processor = processor

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        ex = self.examples[i]
        image = ds_img[ex["idx"]]["image"].convert("RGB")
        W, H = image.size
        boxes = [scale_box(b, W, H) for b in ex["boxes"]]
        labels = [label2id[l] for l in ex["labels"]]
        enc = self.processor(
            image, ex["words"], boxes=boxes, word_labels=labels,
            return_tensors="pt", truncation=True, padding="max_length",
        )
        # processor returns batch dim of 1; squeeze it out
        return {k: v.squeeze(0) for k, v in enc.items()}

train_ds = ReceiptDataset(train_examples, processor)
test_ds  = ReceiptDataset(test_examples,  processor)

# test: pull one item and check shapes
item = train_ds[0]
print("keys:", list(item.keys()))
print("input_ids:", item["input_ids"].shape)
print("labels:", item["labels"].shape)
print("TOTAL tokens in this item:", (item["labels"] == 1).sum().item())

keys: ['input_ids', 'attention_mask', 'bbox', 'labels', 'pixel_values']
input_ids: torch.Size([512])
labels: torch.Size([512])
TOTAL tokens in this item: 4


In [7]:
# NOTEBOOK: 04_model.ipynb  -- cell 6b (find the real out-of-bounds value, on CPU)
bad_label = 0
bad_box = 0
for i in range(len(train_ds)):
    item = train_ds[i]
    lab = item["labels"]
    bbox = item["bbox"]
    uniq = set(lab.tolist())
    if not uniq.issubset({-100, 0, 1}):
        bad_label += 1
        if bad_label <= 3:
            print(f"receipt {i}: unexpected labels {uniq - {-100,0,1}}")
    if bbox.min().item() < 0 or bbox.max().item() > 1000:
        bad_box += 1
        if bad_box <= 3:
            print(f"receipt {i}: bbox range {bbox.min().item()}..{bbox.max().item()}")

print(f"\nreceipts with bad labels: {bad_label}")
print(f"receipts with out-of-range bbox: {bad_box}")

receipt 112: bbox range -2..1006
receipt 143: bbox range 0..1002
receipt 215: bbox range 0..1025

receipts with bad labels: 0
receipts with out-of-range bbox: 4


In [8]:
# NOTEBOOK: 04_model.ipynb  -- cell 6 (dataset wrapper, clamp bbox AFTER processor)
import torch
from torch.utils.data import Dataset
from datasets import load_dataset

ds_img = load_dataset("naver-clova-ix/cord-v2")["train"]
label2id = {"OTHER": 0, "TOTAL": 1}

def scale_box(b, W, H):
    x0, y0, x1, y1 = b
    def clamp(v): return max(0, min(1000, int(v)))
    return [clamp(1000*x0/W), clamp(1000*y0/H), clamp(1000*x1/W), clamp(1000*y1/H)]

class ReceiptDataset(Dataset):
    def __init__(self, examples, processor):
        self.examples = examples
        self.processor = processor
    def __len__(self):
        return len(self.examples)
    def __getitem__(self, i):
        ex = self.examples[i]
        image = ds_img[ex["idx"]]["image"].convert("RGB")
        W, H = image.size
        boxes = [scale_box(b, W, H) for b in ex["boxes"]]
        labels = [label2id[l] for l in ex["labels"]]
        enc = self.processor(
            image, ex["words"], boxes=boxes, word_labels=labels,
            return_tensors="pt", truncation=True, padding="max_length",
        )
        enc = {k: v.squeeze(0) for k, v in enc.items()}
        # SAFETY: clamp bbox tensor to 0..1000 AFTER the processor, so nothing
        # out of range can reach the GPU and trigger the gather assert.
        enc["bbox"] = enc["bbox"].clamp(0, 1000)
        return enc

train_ds = ReceiptDataset(train_examples, processor)
test_ds  = ReceiptDataset(test_examples,  processor)

# verify the fix worked: re-run the out-of-range check
bad = 0
for i in range(len(train_ds)):
    bbox = train_ds[i]["bbox"]
    if bbox.min().item() < 0 or bbox.max().item() > 1000:
        bad += 1
print("datasets ready:", len(train_ds), "train,", len(test_ds), "test")
print("out-of-range bbox receipts now:", bad)

datasets ready: 399 train, 99 test
out-of-range bbox receipts now: 0


In [9]:
# NOTEBOOK: 04_model.ipynb  -- cell 7c (reload model, 2 labels)
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=2
).to("cuda")

print("output classes:", model.config.num_labels)

Loading weights: 100%|██████████| 212/212 [00:02<00:00, 83.77it/s]
[transformers] LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


output classes: 2


In [10]:
# NOTEBOOK: 04_model.ipynb  -- cell 7 (train)
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

device = "cuda"
model.train()

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
class_weights = torch.tensor([1.0, 15.0]).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
optimizer = AdamW(model.parameters(), lr=5e-5)
EPOCHS = 4

for epoch in range(EPOCHS):
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        outputs = model(**batch)
        loss = loss_fn(outputs.logits.reshape(-1, 2), labels.reshape(-1))
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        total_loss += loss.item()
    print(f"epoch {epoch+1}/{EPOCHS}  avg loss: {total_loss/len(train_loader):.4f}")

print("training done")

epoch 1/4  avg loss: 0.2637
epoch 2/4  avg loss: 0.1465
epoch 3/4  avg loss: 0.1233
epoch 4/4  avg loss: 0.0979
training done


In [11]:
# NOTEBOOK: 04_model.ipynb  -- cell 8 (save the trained model)
save_dir = "models/layoutlmv3-total"
model.save_pretrained(save_dir)
processor.save_pretrained(save_dir)
print("saved to", save_dir)

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.14s/it]


saved to models/layoutlmv3-total


In [12]:
# NOTEBOOK: 04_model.ipynb  -- cell 9 (evaluate on held-out test set)
import torch
from torch.utils.data import DataLoader

model.eval()
test_loader = DataLoader(test_ds, batch_size=2)

tp = fp = fn = 0   # for TOTAL class (label 1)
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        logits = model(**batch).logits
        preds = logits.argmax(-1)
        mask = labels != -100          # ignore padding/subword positions
        p = preds[mask]
        t = labels[mask]
        tp += ((p == 1) & (t == 1)).sum().item()
        fp += ((p == 1) & (t == 0)).sum().item()
        fn += ((p == 0) & (t == 1)).sum().item()

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall    = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
print(f"TOTAL  precision: {precision:.3f}   recall: {recall:.3f}   F1: {f1:.3f}")
print(f"(tp={tp}, fp={fp}, fn={fn})")

TOTAL  precision: 0.808   recall: 0.963   F1: 0.879
(tp=181, fp=43, fn=7)
